In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np



In [2]:
model = load_model('model-1.h5')

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

with open('onehot_encoder_geo.pkl','rb') as file:
    onehot_encoder_geo = pickle.load(file)



In [3]:
input_data = {
    'CreditScore':630,
    'Geography':'France',
    'Gender':'Female',
    'Age':40,
    'Tenure':3,
    'Balance':600000,
    'NumOfProducts':1,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':101348.88
}

In [4]:
onehot_encoded_geo = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()

geo_encoded_df = pd.DataFrame(onehot_encoded_geo, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))



c:\Users\arjun\Documents\Lang_chain_learning\DeepLearning\ANN model\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [5]:
geo_encoded_df


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [6]:
input_data_df = pd.DataFrame([input_data])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,630,France,Female,40,3,600000,1,1,1,101348.88


In [7]:
input_data_df['Gender'] = label_encoder_gender.transform(input_data_df['Gender'])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,630,France,0,40,3,600000,1,1,1,101348.88


In [8]:
input_data_df = pd.concat([input_data_df.drop('Geography',axis=1), geo_encoded_df],axis=1)
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,630,0,40,3,600000,1,1,1,101348.88,1.0,0.0,0.0


In [9]:
input_data_df_scaled = scaler.transform(input_data_df)
input_data_df_scaled

array([[-0.22465323, -1.09499335,  0.10479359, -0.69539349,  8.38812313,
        -0.91668767,  0.64920267,  0.97481699,  0.01595384,  1.00150113,
        -0.57946723, -0.57638802]])

In [10]:
prediction = model.predict(input_data_df_scaled)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step
[[0.9805662]]


In [11]:
prediction_probab = prediction[0][0]
num = np.float32(prediction_probab)
print(num)

0.9805662


In [12]:
if num>0.5:
    print('Customer is likely to churn')

else:
    print('Customer is not likely to churn')

Customer is likely to churn
